# Applied ML Workshop — Pneumonia Detection

**Duration:** ~1.5 hours
**Before you start:**
1. The data used are from the [Chest X-Ray Pneumonia dataset](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia).


---
## Block 1 — ML workflow & problem framing (~10 min)

*Discussion: read this section before running code.*

**Machine learning** is the process of learning patterns from data so a system can perform tasks on new, unseen inputs — such as predicting a value, classifying an observation, or revealing hidden structure.

| Type                       | Analogy                               | What the model learns                             |
| -------------------------- | ------------------------------------- | ------------------------------------------------- |
| **Supervised learning**    | Fruit with labels ("apple", "banana") | Map inputs to known outputs from labeled examples |
| **Unsupervised learning**  | Unlabeled mixed fruit                 | Group similar items without predefined categories |
| **Reinforcement learning** | Puppy rewarded for fetching apples    | Optimal actions through trial, error, and rewards |


The ML workflow below shows how these steps apply in practice.

| Step | What you do | Example (pneumonia workshop) |
|------|-------------|------------------------------|
| 1. **Define problem** | State the real-world goal and a measurable proxy metric | Goal: detect pneumonia; metric: recall on pneumonia class |
| 2. **Prepare data** | Load, clean, explore, split, preprocess | Load X-rays; normalize pixels; train/val/test split |
| 3. **Choose model** | Match architecture to data type; start simple | CNN for images; linear regression for tabular baseline |
| 4. **Train** | Minimize loss by updating model weights | Forward → loss → backward → optimizer step |
| 5. **Validate** | Monitor performance on held-out validation data | Track val accuracy each epoch; tune learning rate |
| 6. **Test & report** | Evaluate once on test set; report metrics and failures | Confusion matrix; inspect misclassified X-rays |
| 7. **Iterate** | Loop back if performance or requirements change | Add augmentation, change architecture, collect more data |


```mermaid
flowchart LR
    A[Define problem] --> B[Prepare data]
    B --> C[Choose model]
    C --> D[Train]
    D --> E[Validate]
    E --> F{Good enough?}
    F -->|No| C
    F -->|No| B
    F -->|Yes| G[Test and report]
    G --> H[Deploy or publish]
    H -->|New data| B
```

**Critical rule:** the **test set** is touched only once, at the end. Validation guides tuning; testing estimates real-world performance.

> **Checkpoint:** For a climate prediction project, what would be a real-world goal and a proxy metric?



### Essential concepts

- **Supervised learning** — learn from labeled examples (`PNEUMONIA` / `NORMAL` folders).
- **Model** — parameterized function (CNN weights).
- **Loss** — scalar to minimize; wrong predictions increase loss.
- **Optimizer** — updates weights using gradients from the loss.
- **Epoch** — one full pass through the training set.
- **Train / val / test** — train to learn; val to monitor and tune; test to estimate real-world performance.




---
## Block 2 — Load & explore data (~20 min)

Set paths and load the chest X-ray dataset. Inspect class balance and sample images.


In [ ]:
# --- Configuration ---
# Path to dataset root (must contain train/, val/, test/ subfolders)
DATA_ROOT = '/scratch/vp91/zxw900/applied_ml/data/chest_xray'
# Workshop demo: limit images per class for faster runs (set None for full dataset)
MAX_SAMPLES_PER_CLASS = 50

# Training epochs (use 4–6 for a 2-hour session on CPU; 12 for full run)
NUM_EPOCHS = 6

BATCH_SIZE = 32
IMG_SIZE = 150
LABELS = ['PNEUMONIA', 'NORMAL']


In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


### Dataset background

The dataset contains 5,863 pediatric chest X-rays images (JPEG) in `train/`, `val/`, and `test/` folders, each with `PNEUMONIA` and `NORMAL` subfolders. Images were quality-controlled and labeled by expert physicians before being released for ML training.


In [ ]:
labels = ['PNEUMONIA', 'NORMAL']

size_counts = {}
for label in labels:
    path = os.path.join(DATA_ROOT, 'train', label)
    for img_name in os.listdir(path)[:50]:
        img_arr = cv2.imread(os.path.join(path, img_name), cv2.IMREAD_GRAYSCALE)
        if img_arr is not None:
            h, w = img_arr.shape[:2]
            size_counts[(h, w)] = size_counts.get((h, w), 0) + 1

print('Original image sizes (height x width) in train set:')
for (h, w), count in sorted(size_counts.items(), key=lambda x: -x[1]):
    print(f'  {h} x {w}: {count} images')

if size_counts:
    heights = [h for (h, w) in size_counts for _ in range(size_counts[(h, w)])]
    widths = [w for (h, w) in size_counts for _ in range(size_counts[(h, w)])]
    print(f'\nAll train images — height range: {min(heights)}–{max(heights)}, width range: {min(widths)}–{max(widths)}')

In [ ]:
def get_training_data(data_dir, max_per_class=MAX_SAMPLES_PER_CLASS):
    """Load images as (array, label) pairs. Labels: 0=PNEUMONIA, 1=NORMAL."""
    data = []
    for label in LABELS:
        path = os.path.join(data_dir, label)
        class_num = LABELS.index(label)
        filenames = sorted(os.listdir(path))
        if max_per_class is not None:
            filenames = filenames[:max_per_class]
        for img_name in filenames:
            img_path = os.path.join(path, img_name)
            img_arr = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img_arr is None:
                continue
            resized = cv2.resize(img_arr, (IMG_SIZE, IMG_SIZE))
            data.append([resized, class_num])
    return np.array(data, dtype=object)


train = get_training_data(os.path.join(DATA_ROOT, 'train'))
val = get_training_data(os.path.join(DATA_ROOT, 'val'))
test = get_training_data(os.path.join(DATA_ROOT, 'test'))

print(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')


In [ ]:
# Class distribution in training set (note imbalance)
class_names = ['Pneumonia' if item[1] == 0 else 'Normal' for item in train]

sns.set_style('darkgrid')
sns.countplot(x=class_names)
plt.title('Training set class counts')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()


In [ ]:
# Preview one image from each class
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, idx in zip(axes, [0, -1]):
    ax.imshow(train[idx][0], cmap='gray')
    ax.set_title(LABELS[train[idx][1]])
    ax.axis('off')
plt.tight_layout()
plt.show()


> **Checkpoint:** Why do we keep a separate test set until the final evaluation?


### Choosing Architectures based on data
- Tabular measurements → **MLP**
- 2D/3D images → **CNN** (classify) or **U-Net** (segment)
- Time-ordered data → **RNN / LSTM**
- Networks of entities → **GNN**


| Architecture | Strength                       | Limitation                          |
| ------------ | ------------------------------ | ----------------------------------- |
| MLP          | Simple, fast on tabular data   | Ignores spatial/temporal structure  |
| CNN          | Strong on images and grids     | Needs more data than linear models  |
| RNN/LSTM     | Captures temporal dependencies | Harder to train on long sequences   |
| GNN          | Models irregular relationships | Requires careful graph construction |
| U-Net        | Precise pixel-level outputs    | Needs detailed mask annotations     |


A **neural network** is a stack of **neurons** organized in **layers**. Each neuron applies weights and a bias, then an **activation function**. Training adjusts the weights to minimize a **loss function**.

Linear regression is the simplest case: one weighted sum plus bias. Deep networks stack many such operations with nonlinear **activation functions**.

<p align="center">
  <img src="https://intro-to-pytorch.readthedocs.io/en/latest/_images/NN_first.png" alt="A simple neural network" width="50%"/>
</p>

<p align="center"><em> A simple feedforward neural network.</em></p>


| Architecture | Best for | Science use cases |
|--------------|----------|-------------------|
| **MLP** | Tabular / vector data | Surrogate models; material property prediction |
| **CNN** | Images and spatial grids | X-rays; microscopy; climate fields |
| **RNN/LSTM/Transformers** | Sequences and time series | Sensor logs; weather records; genomic sequences |
| **GNN** | Graph-structured data | Molecules; materials; simulation meshes |
| **U-Net** | Image segmentation | Cell boundaries; tumor regions in microscopy |

#### MLP — Multilayer Perceptron

<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/4/47/MultiLayerNeuralNetwork_english.png" alt="MLP architecture" width="55%"/>
</p>

<p align="center"><em> MLP architecture. Source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:MultiLayerNeuralNetwork_english.png).</em></p>


**Data flow:** input vector → **Linear** (weighted sum + bias) → **Activation** (ReLU) → … → output

| Layer | Data in → Data out |
|-------|-------------------|
| **Input** | Feature vector (e.g. temperature, pressure, composition) |
| **Fully connected (`nn.Linear`)** | Each output neuron combines *all* inputs with learned weights |
| **Activation (ReLU)** | Applies nonlinearity; without it, stacked layers collapse to one linear map |
| **Output** | Single value (regression) or class scores (classification) |

Every input connects to every neuron in the next layer — good for tabular data, but ignores spatial or sequential structure.

**Data input examples**

| | Shape / format | Example |
|---|----------------|---------|
| **Usual** | `(n_features,)` per sample | Credit scoring: `[income, debt_ratio, payment_history_score, …]` |
| **Scientific** | `(n_features,)` per sample | Surrogate model: `[porosity, grain_size, sinter_temp, …]` → predict conductivity; simulation parameters in, stress field summary out |

#### CNN — Convolutional Neural Network

<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png" alt="CNN architecture" width="65%"/>
</p>

<p align="center"><em> Typical CNN architecture. Source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Typical_cnn.png).</em></p>


**Data flow:** image tensor → **Conv** → **ReLU** → **Pool** → … → **Flatten** → **Linear** → prediction

| Layer | Data in → Data out |
|-------|-------------------|
| **Conv2d** | A small filter slides across the image; each position produces one value in a **feature map** (local patterns: edges, textures) |
| **ReLU** | Zeroes negative activations; keeps computation sparse and nonlinear |
| **MaxPool2d** | Downsamples each region (e.g. 2×2 window → max value); reduces spatial size, builds translation tolerance |
| **BatchNorm** | Normalizes activations per channel; stabilizes training |
| **Dropout** | Randomly drops units during training; reduces overfitting |
| **Flatten + Linear** | Converts 2D feature maps to a vector, then classifies |

Spatial structure is preserved until pooling — why CNNs excel on images and gridded scientific fields. This workshop builds a `PneumoniaCNN` below.

**Data input examples**

| | Channels × spatial size | Example |
|---|---------------------------|---------|
| **Usual** | `3 × 224 × 224` | Natural-image classification (RGB) |
| **Scientific** | `1 × 150 × 150` | Single-channel chest X-ray (grayscale) |
| **Scientific** | `C × H × W`, **C ≫ 3** | **Weather / climate CNN:** each channel is a 2D field — surface temperature, specific humidity, geopotential height, zonal/meridional wind at several pressure levels, plus past and future time slices. A single input tensor can have **hundreds to thousands of channels** while still using the same Conv2d → Pool workflow as a photo CNN |
| **Scientific** | `C × H × W` | Microscopy or satellite **multispectral** stack: bands beyond visible RGB (e.g. NIR, thermal) |

The key idea: **channels are generic feature maps**. In vision they are R, G, B; in weather they are physical variables aligned on the same grid.


**RNN**
<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/b/b5/Recurrent_neural_network_unfold.svg" alt="RNN unrolled over time" width="65%"/>
</p>

<p align="center"><em> RNN folded (left) and unrolled across time steps (right). Source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Recurrent_neural_network_unfold.svg).</em></p>


**GNN**
<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/ec/Message_Passing_Neural_Network.png" alt="GNN message passing" width="60%"/>
</p>

<p align="center"><em> Message passing between graph nodes. Source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Message_Passing_Neural_Network.png).</em></p>


**U-Net**
<p align="center">
  <img src="https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/u-net-architecture.png" alt="U-Net architecture" width="62%"/>
</p>

<p align="center"><em> U-Net encoder–decoder with skip connections. Source: [U-Net paper (Ronneberger et al., 2015)](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/).</em></p>


### Input shapes

CNNs are not limited to 3-channel RGB photos. In science, **channels often represent physical variables** stacked over the same spatial grid — the conv layers still scan locally, but each "pixel" carries many measurements.

| Architecture | Usual input example | Scientific input example |
|--------------|---------------------|--------------------------|
| **MLP** | Customer record: `[age, income, spend, …]` → vector shape `(n_features,)` | Experiment row: `[temperature, pressure, alloy %, …]` → predict yield strength |
| **CNN** | RGB photo: `(3, 224, 224)` — 3 colour channels | Grayscale X-ray: `(1, 150, 150)` — 1 channel (this notebook) |
| **CNN** | — | Weather / climate **field stack**: `(C, H, W)` with **hundreds or thousands of channels** — e.g. temperature, humidity, pressure, wind $(u, v)$ at multiple heights and forecast lead times on a lat–lon grid |
| **RNN / LSTM** | Daily stock price sequence: `(time_steps, 1)` | Hourly sensor log: `(time_steps, n_sensors)` — temperature, wind, rainfall at a station |
| **GNN** | Social network: users = nodes, friendships = edges | Molecule: atoms = nodes (element, charge), bonds = edges; or FEM mesh nodes and elements |
| **U-Net** | RGB street image `(3, H, W)` → per-pixel class mask | Multispectral satellite tile or microscopy stack → per-pixel cell-type or land-cover mask |

**Tensor layout reminder (PyTorch CNN):** `(batch, channels, height, width)`. More channels = more variables at each spatial location, not necessarily more colours.



---
## Block 3 — Preprocess & augment (~20 min)

What might happen if we use the imbalanced dataset for training?

In order to increase the generalization, we can expand artificially our dataset. The idea is to alter the training data with small transformations to reproduce the variations.
Approaches that alter the training data in ways that change the array representation while keeping the label the same are known as **data augmentation** techniques. Some popular augmentations people use are grayscales, horizontal flips, vertical flips, random crops, color jitters, translations, rotations, and much more.


Convert images to normalized tensors and build PyTorch `DataLoader`s. Apply augmentation on **training data only**.


In [ ]:
def unpack_split(data):
    x, y = [], []
    for feature, label in data:
        x.append(feature)
        y.append(label)
    return np.array(x, dtype=np.float32), np.array(y, dtype=np.float32)


x_train, y_train = unpack_split(train)
x_val, y_val = unpack_split(val)
x_test, y_test = unpack_split(test)

# Scale pixel values to [0, 1] for stable training
x_train /= 255.0
x_val /= 255.0
x_test /= 255.0

print(f'x_train shape: {x_train.shape}, pixel range: [{x_train.min():.2f}, {x_train.max():.2f}]')


For the data augmentation:
1. Randomly rotate some training images by 30 degrees
2. Randomly Zoom by 20% some training images
3. Randomly shift images horizontally by 10% of the width
4. Randomly shift images vertically by 10% of the height
5. Randomly flip images horizontally.

In [ ]:
class PneumoniaDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).unsqueeze(0)
        return image, torch.tensor(label, dtype=torch.float32)


train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

train_dataset = PneumoniaDataset(x_train, y_train, transform=train_transform)
val_dataset = PneumoniaDataset(x_val, y_val, transform=eval_transform)
test_dataset = PneumoniaDataset(x_test, y_test, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


| Technique | Why |
|-----------|-----|
| Normalize / 255 | Stable gradients; faster CNN convergence |
| Augmentation (train only) | Expand effective dataset; reduce overfitting |
| `Dataset` + `DataLoader` | Batched, shuffled input to the training loop |
| `shuffle=True` (train) | Different mini-batches each epoch |

> **Checkpoint:** Why do we divide by 255? Why not augment the test set?


---
## Block 4 — Build, train & test (~45 min)


Define the CNN, loss, and optimizer; run the training loop; evaluate on the test set.



### Building blocks

| Concept | What it means | Science example |
|---------|---------------|-----------------|
| **Input layer** | Raw features fed to the network | Temperature, pressure, pixel values |
| **Hidden layers** | Intermediate representations learned from data | Edges → textures → structures in an X-ray |
| **Output layer** | Final prediction | House price, pneumonia probability |
| **Fully connected layer** | Every input connects to every output | Tabular regression on experiment measurements |
| **Activation function** | Nonlinearity enabling complex patterns | ReLU, sigmoid |
| **Loss function** | Error measure to minimize | MSE (regression); BCE / cross-entropy (classification) |
| **Forward pass** | Input flows through layers → prediction | Model scores an X-ray for pneumonia |
| **Backward pass** | Gradients computed to update weights | `loss.backward()` in PyTorch |
| **Optimizer** | Rule for weight updates | SGD, Adam, RMSprop |
| **Learning rate** | Step size per update | Too large → unstable; too small → slow |
| **Batch / epoch** | Subset per update / one full pass through training data | 32 images per batch, 12 epochs |
| **Regularization** | Reduce overfitting | Dropout, batch norm, data augmentation |

### PyTorch workflow (preview)

1. **Structure** — `nn.Module` with layers in `__init__` and logic in `forward`
2. **Data** — `Dataset` → `DataLoader` → mini-batches
3. **Train** — forward → loss → `backward()` → `optimizer.step()`
4. **Evaluate** — `model.eval()`, no gradients, metrics on val/test sets

**One training batch:**
1. Forward pass → model outputs
2. Compute loss → compare outputs to labels
3. Backward pass → gradients via autograd
4. `optimizer.step()` → update weights


In [ ]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            # nn.Dropout(0.1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            # nn.Dropout(0.2),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            # nn.Dropout(0.2),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            # nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x).squeeze(1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PneumoniaCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.RMSprop(model.parameters())
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=2, min_lr=1e-6)

print(model)
print(f'Using device: {device}')


**CNN — Convolutional Neural Network**

```{figure} https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png
:alt: CNN architecture
:width: 560px
:align: center

Typical CNN architecture. Source: [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Typical_cnn.png).
```

**Data flow:** image tensor → **Conv** → **ReLU** → **Pool** → … → **Flatten** → **Linear** → prediction


| Layer                | Data in → Data out                                                                                                              |
| -------------------- | ------------------------------------------------------------------------------------------------------------------------------- |
| **Conv2d**           | A small filter slides across the image; each position produces one value in a **feature map** (local patterns: edges, textures) |
| **ReLU**             | Zeroes negative activations; keeps computation sparse and nonlinear                                                             |
| **MaxPool2d**        | Downsamples each region (e.g. 2×2 window → max value); reduces spatial size, builds translation tolerance                       |
| **BatchNorm**        | Normalizes activations per channel; stabilizes training                                                                         |
| **Dropout**          | Randomly drops units during training; reduces overfitting                                                                       |
| **Flatten + Linear** | Converts 2D feature maps to a vector, then classifies                                                                           |


Spatial structure is preserved until pooling — why CNNs excel on images and gridded scientific fields. See `PneumoniaCNN` in `[workshop.ipynb](workshop.ipynb)`.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        if is_train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        if is_train:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += images.size(0)

    return running_loss / total, correct / total


history = {'accuracy': [], 'loss': [], 'val_accuracy': [], 'val_loss': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)
    scheduler.step(val_acc)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(
        f'Epoch {epoch + 1}/{NUM_EPOCHS} - '
        f'loss: {train_loss:.4f} - acc: {train_acc:.4f} - '
        f'val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}'
    )


In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc * 100:.2f}%')


> **Checkpoint:** Name the four steps inside one training batch.


---
## Block 5 — Evaluate & wrap-up (~25 min)

Plot learning curves, report per-class metrics, and inspect prediction errors.


In [ ]:
epochs = list(range(1, NUM_EPOCHS + 1))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(epochs, history['accuracy'], 'go-', label='Training')
ax[0].plot(epochs, history['val_accuracy'], 'ro-', label='Validation')
ax[0].set_title('Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(epochs, history['loss'], 'g-o', label='Training')
ax[1].plot(epochs, history['val_loss'], 'r-o', label='Validation')
ax[1].set_title('Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)

predictions = np.array(all_preds)
print(classification_report(
    y_test, predictions,
    target_names=['Pneumonia (Class 0)', 'Normal (Class 1)'],
))


In [ ]:
cm = confusion_matrix(y_test, predictions)
cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, cmap='Blues', annot=True, fmt='d', linewidths=1, linecolor='black')
plt.title('Confusion matrix (test set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
def plot_examples(indices, title):
    n = min(6, len(indices))
    if n == 0:
        print(f'No examples for: {title}')
        return
    fig, axes = plt.subplots(2, 3, figsize=(9, 6))
    fig.suptitle(title)
    for ax, idx in zip(axes.flat, indices[:n]):
        ax.imshow(x_test[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
        ax.set_title(f'Pred {predictions[idx]}, Actual {int(y_test[idx])}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


correct = np.where(predictions == y_test)[0]
incorrect = np.where(predictions != y_test)[0]

plot_examples(correct, 'Correctly classified')
plot_examples(incorrect, 'Incorrectly classified')


## Summary
### Model selection checklist

1. Define problem and metric first.
2. Start with a clear baseline.
3. Respect train / val / test splits.
4. Monitor train vs val gap for overfitting.
5. Report test metrics and show failure cases.

## ML Workflow

| Step | This notebook |
|------|---------------|
| **Define problem** | Binary classification: pneumonia vs normal on chest X-ray |
| **Choose metric** | Accuracy, precision/recall per class (false negatives may be clinically costly) |
| **Prepare data** | Load JPEGs, resize, normalize, augment, batch with `DataLoader` |
| **Choose model** | CNN (`PneumoniaCNN`) — images need spatial feature learning |
| **Train** | Minimize `BCEWithLogitsLoss` with `RMSprop` over epochs |
| **Validate** | Monitor val loss/accuracy each epoch; scheduler adjusts learning rate |
| **Test** | Final metrics on held-out test set — **once**, at the end |
| **Report** | Plots, confusion matrix, inspect misclassified images |


### Take-home

- [`bonus-finetuning.ipynb`](bonus-finetuning.ipynb) — transfer learning with ResNet-18.

> **Closing:** Tune the model in your own time.
